# Markov chains for text generation

A Markov chain describes a system whose states change over time.

The changes are governed by a probability distribution. The **next state only depends upon the current state of the system**.

Voglio stimare la probabilità di transizione da uno stato all'altro.

Markov chains have many applications in scientific computing, given their ability to model complex processes and make predictions based on limited information:

* genemark algorithm for gene prediction
* the Metropolis algorithm for measuring thermodynamical properties
* Google's PageRank algorithm for Web search
* speech recognition
* handwriting recognition
* data compression
* spam filtering

## $K^{th}$ order Markov models

In a 1948 paper A Mathematical Theory of Communication, Claude Shannon proposed using a Markov chain to **create a statistical model of the sequences of letters in a piece of English text**, laying the groundwork for today's Information Age.

We say that $p(X_1, X_2, \dots , X_{j+k-1})$ is $k^{th}$ order Markov for $k>0$ if:

$
    p(X_{j+k} | X_1, X_2, \dots , X_{j+k-1}) = p(X_{j+k} | X_j, X_{j+1}, \dots , X_{j+k-1}) 
$

for any $j$ such that $j + k \leq n$.

Today **we will implement and test different $k^{th}$ order Markov models** to see how and how well they reproduce English language features.

In [60]:
file = './04_great_expectations_cleaned.txt'

with open(file, 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:600])




My father’s family name being Pirrip, and my Christian name Philip, my
infant tongue could make of both names nothing longer or more explicit
than Pip. So, I called myself Pip, and came to be called Pip.

I give Pirrip as my father’s family name, on the authority of his
tombstone and my sister,—Mrs. Joe Gargery, who married the blacksmith.
As I never saw my father or my mother, and never saw any likeness of
either of them (for their days were long before the days of
photographs), my first fancies regarding what they were like were
unreasonably derived from their tombstones. The shape of the 


Vogliamo usare questo testo per stimare la probabilità di avere una data parola dopo un'altra.

Per il modello *my* e *My* sono stringhe diverse. Quindi prima di tutto elimino alcune feature dal testo, come la punteggiatura o le lettere maiuscole, in modo da semplificare il testo.

## Which are the building blocks of a language?

Let's start by considering characters.

We want to merge all the text into a single stream of printable characters.

To do this we will use `itertools.chain.from_iterable` to combine lines into a single stream.

In [61]:
import itertools as it
import random as rn

l = [['hello', 'world'],['nice', 'day']]
l1 = it.chain.from_iterable(l)
for el in l1:
    print(el)

hello
world
nice
day


Posso anche estrarre tutti i caratteri del testo:

In [62]:
l = 'hello world'
l1 = it.chain.from_iterable(l)
for el in l1:
    print(el)

h
e
l
l
o
 
w
o
r
l
d


We want to remove also all characters that are diffrent from letters.

In [63]:
import re

text = "Hello, world! This is a test."

tt = re.sub(r'[^\w\s]', '', text)

print(tt)

Hello world This is a test


Apriamo il file e estraiamo tutti i caratteri per ogni riga del testo. Convertiamo tutti i caratteri in minuscolo.

In [64]:
file = './04_great_expectations_cleaned.txt'
with open(file, 'r', encoding='utf8') as text:
    lines = (re.sub(r'[^\w\s]', '', line).lower() for line in text)
    characters = it.chain.from_iterable(lines)
    result = list(characters)

print(result[:30])

['\n', '\n', 'm', 'y', ' ', 'f', 'a', 't', 'h', 'e', 'r', 's', ' ', 'f', 'a', 'm', 'i', 'l', 'y', ' ', 'n', 'a', 'm', 'e', ' ', 'b', 'e', 'i', 'n', 'g']


Ci sono ancora spazi `' '` e line break `'\n'`.

In English, the probability of a letter depends strongly on the previous few letters but only weakly on the previous 50 letters.

Se continuo ad aumentare la memoria sto riducendo la statistica.

## First order Markov chains
Let's generate each letter by following these steps:

* look at the last letter in the text that has been generated so far;
* consider in the original text how often each letter follows the chosen one;
* generate the new letter based on this conditional probability;
* consider the last letter generated as new starting point and go back to point 2.

Vediamo quali sono i caratteri più e meno comuni. 

In [65]:
from collections import Counter

counts = Counter(result)
print(counts)

Counter({' ': 168157, 'e': 91702, 't': 68614, 'a': 62849, 'o': 59867, 'i': 54474, 'n': 52701, 'h': 48082, 's': 45132, 'r': 40780, 'd': 36630, 'l': 28004, 'm': 22549, 'u': 21712, '\n': 20244, 'w': 20232, 'c': 16878, 'g': 16441, 'y': 16061, 'f': 15742, 'p': 12851, 'b': 12167, 'k': 7535, 'v': 6718, 'j': 1669, 'x': 938, 'q': 699, '_': 494, 'z': 164, '2': 3, '4': 1, '\t': 1, 'ô': 1, 'ê': 1, '1': 1})


Let's normalize the uncommon letters:

In [66]:
to_replace = {'ô': 'o', 'ê': 'i'}

def letter_normalization(letter):
    if letter in to_replace:
        return to_replace[letter] 
    return letter

norm_result = []

with open(file, 'r', encoding='utf8') as text:
    lines = (re.sub(r'[^\w\s]', '', line).lower() for line in text)
    characters = it.chain.from_iterable(lines)
    result = list(characters)
    for letter in result:
        modified_letter = letter_normalization(letter)
        norm_result.append(modified_letter)

counts = Counter(norm_result)
print(counts)

len(counts)

Counter({' ': 168157, 'e': 91702, 't': 68614, 'a': 62849, 'o': 59868, 'i': 54475, 'n': 52701, 'h': 48082, 's': 45132, 'r': 40780, 'd': 36630, 'l': 28004, 'm': 22549, 'u': 21712, '\n': 20244, 'w': 20232, 'c': 16878, 'g': 16441, 'y': 16061, 'f': 15742, 'p': 12851, 'b': 12167, 'k': 7535, 'v': 6718, 'j': 1669, 'x': 938, 'q': 699, '_': 494, 'z': 164, '2': 3, '4': 1, '\t': 1, '1': 1})


33

Abbiamo 33 caratteri.

We need to take our characters in couples.

If we have a text like "home", we want to obtain the following couples:

(h, o) (o, m) (m, e)

Costruiamo un bigram, tutte le coppie di lettere:

In [67]:
bigrams = []

for i in range(len(norm_result) - 1):
    bigrams.append((norm_result[i], norm_result[i+1]))

bigram_counts = Counter(bigrams)
print(len(bigram_counts))

bigram_counts.most_common(20)


653


[(('e', ' '), 30661),
 ((' ', 't'), 23221),
 (('d', ' '), 22224),
 (('t', 'h'), 20551),
 ((' ', 'a'), 19921),
 (('h', 'e'), 19625),
 (('t', ' '), 19224),
 (('s', ' '), 15742),
 ((' ', 'i'), 14392),
 (('i', 'n'), 14071),
 ((' ', 'h'), 13572),
 (('a', 'n'), 13003),
 (('e', 'r'), 12985),
 ((' ', 'w'), 12899),
 (('n', ' '), 12365),
 ((' ', 's'), 11626),
 (('n', 'd'), 10748),
 (('h', 'a'), 10447),
 ((' ', 'o'), 10078),
 (('r', ' '), 10039)]

Questi sono i bigram più comuni. Ad esempio emergono delle coppie `th` o `er`.

In order to generate the new letter starting from the previous one, we have to select only the part of our dictionary that contains our letter as initial part.

We can use the dictionary `items` function to get the sequence of key-value pairs, and filter them by their starting point.

In [68]:
def starts_with(sequence, letter):
    return sequence[0]==letter

# creo un dizionario di bigrammi fittizi con relativa frequenza
fake_counts = {('a', 'b'): 1, ('b', 'c'): 2, ('a', 'p'): 3}
letter = 'a'
ks = [k for k, v in fake_counts.items() if starts_with(k, letter)]
print(ks)

letter = 'a'
letters = [k[-1] for k, v in fake_counts.items() if starts_with(k, letter)]
occurrences = [v for k, v in fake_counts.items() if starts_with(k, letter)]
print(letters, occurrences)

[('a', 'b'), ('a', 'p')]
['b', 'p'] [1, 3]


To generate our text, we have to start from an initial character.

Usiamo ad esempio lo spazio come primo elemento. Generiamo un testo di 100 caratteri:

In [70]:
new_text_ = [' ']

for i in range(100):
    character = new_text_[-1]
    characters = [k[-1] for k, v in count_couples.items() if starts_with(k, character)]
    occurrences = [v for k, v in count_couples.items() if starts_with(k, character)]
    next_character = rn.choices(characters, occurrences)[0]
    new_text_.append(next_character)
    
new_text = str.join('', new_text_)

print(new_text)


NameError: name 'count_couples' is not defined

Let's try to generate a longer text.

In [ ]:
new_text_ = ['t']
for i in range(500):
    character = new_text[-1]
    characters = [k[-1] for k, v in count_couples.items() if starts_with(k, character)]
    occurrences = [v for k, v in count_couples.items() if starts_with(k, character)]
    next_character = rn.choices(characters, occurrences)[0]
    new_text_.append(next_character)
    
new_text = str.join('', new_text_)

print(new_text)

I risultati sono deludenti. Non ritroviamo le feature della lingua.

Aumentiamo la memoria, invece di usare bigram usiamo dei trigram.

Let's consider $2^{nd}$ order Markov chains

In [ ]:
trigrams = []
for i in range(len(norm_result) - 2):
    trigram = tuple(norm_result[i:i+3])
    trigrams.append(trigram)
        
trigram_counts = Counter(trigrams)
print(len(trigram_counts))

trigram_counts.most_common(20)

6336


[((' ', 't', 'h'), 14699),
 (('t', 'h', 'e'), 11605),
 (('h', 'e', ' '), 10240),
 (('a', 'n', 'd'), 8130),
 (('n', 'd', ' '), 7961),
 ((' ', 'a', 'n'), 7620),
 (('e', 'd', ' '), 6837),
 (('i', 'n', 'g'), 6072),
 ((' ', 't', 'o'), 5913),
 (('n', 'g', ' '), 5647),
 (('a', 't', ' '), 5415),
 ((' ', 'i', ' '), 5266),
 (('t', 'o', ' '), 5195),
 (('e', 'r', ' '), 5072),
 ((' ', 'h', 'a'), 4631),
 ((' ', 'o', 'f'), 4591),
 ((' ', 'h', 'e'), 4548),
 (('a', 's', ' '), 4475),
 (('o', 'f', ' '), 4097),
 ((' ', 'i', 'n'), 4095)]

Emergono altre strutture come `ing` o `the`.

In [ ]:
def starts_with(sequence, context):
    return sequence[:len(context)] == context

new_text_ = ['t', 'h']  

for i in range(500):
    context = tuple(new_text_[-2:])  

    characters = [
        k[-1] for k, v in trigram_counts.items()
        if starts_with(k, context)
    ]

    occurrences = [
        v for k, v in trigram_counts.items()
        if starts_with(k, context)
    ]

    if not characters:
        break  

    next_character = rn.choices(characters, occurrences)[0]
    new_text_.append(next_character)

new_text = ''.join(new_text_)

print(new_text)

the heturn thave
ifer and mucerent of twought the briked recion ass a leso we pont low wayin the of yougave shis wer yought i diss ned beterbe hen ifurn a reard the usext forne gled hersise shess geir thad abled
he
hat giblund wed oustemin qualiked the his star ats of tur baress to stang so
laidight me
den
ted too shearry th a
ling afted had wend and younly lit a bliellacted ral youbmisery i hander in ling on shard gookin by on
hat hit forawassis of don muchmed as acknot i

i know somparkedithippo


In [ ]:
new_text_ = ['a', 't']  

for i in range(500):
    context = tuple(new_text_[-2:])  

    characters = [
        k[-1] for k, v in trigram_counts.items()
        if starts_with(k, context)
    ]

    occurrences = [
        v for k, v in trigram_counts.items()
        if starts_with(k, context)
    ]

    if not characters:
        break  

    next_character = rn.choices(characters, occurrences)[0]
    new_text_.append(next_character)

new_text = ''.join(new_text_)

print(new_text)

Ora ci sono delle parole che esistono e la struttura richiama un po' la lingua inglese.

I caratteri iniziali sono importanti.

Let's now **consider words ad building blocks**.

In [73]:
with open(file, 'r', encoding='utf-8') as f:
    text = f.read().lower()
    words = re.findall(r'\b[a-zàèéìòù]+\b', text)

print(words[:10])

from collections import Counter

word_counts = Counter(words)

word_counts.most_common(20)

['my', 'father', 's', 'family', 'name', 'being', 'pirrip', 'and', 'my', 'christian']


[('the', 8143),
 ('and', 7097),
 ('i', 6625),
 ('to', 5157),
 ('of', 4438),
 ('a', 4054),
 ('that', 3062),
 ('in', 3028),
 ('was', 2822),
 ('it', 2796),
 ('you', 2274),
 ('he', 2247),
 ('had', 2092),
 ('my', 2061),
 ('me', 1989),
 ('his', 1854),
 ('as', 1775),
 ('with', 1759),
 ('at', 1639),
 ('on', 1421)]

Dato che ho aumentato la memoria del modello (passando da lettere a parole) vediamo cosa accade con modello Markov Chain di ordine 0: quindi senza memoria delle parole precedenti.

In [80]:
import random as rn 

random_sample = rn.choices(list(word_counts.keys()), weights=list(word_counts.values()), k=30)

new_text = str.join(' ', random_sample)

print(new_text)

and evidently far cheerful and as his more capable me now be in great seems such we it and a devil the i mr her before will you as night


Non ha senso ma almeno è leggibile.

Passiamo al primo ordine:

In [81]:
bigrams = []

for i in range(len(words) - 1):
    bigrams.append((words[i], words[i+1]))

bigram_counts = Counter(bigrams)
print(len(bigram_counts))

bigram_counts.most_common(20)


79360


[(('of', 'the'), 842),
 (('in', 'the'), 785),
 (('i', 'had'), 628),
 (('that', 'i'), 566),
 (('i', 'was'), 499),
 (('and', 'i'), 491),
 (('it', 'was'), 471),
 (('to', 'be'), 438),
 (('on', 'the'), 427),
 (('to', 'the'), 414),
 (('at', 'the'), 411),
 (('in', 'a'), 333),
 (('when', 'i'), 313),
 (('miss', 'havisham'), 310),
 (('and', 'the'), 308),
 (('with', 'a'), 307),
 (('to', 'me'), 287),
 (('don', 't'), 285),
 (('that', 'he'), 284),
 (('he', 'had'), 279)]

In [ ]:
prob = {}

for (w1, w2), count in bigram_counts.items():
    prob[(w1, w2)] = count / word_counts[w1]

print("prob", prob[('in', 'the')])

print("bigram_counts", bigram_counts[('in', 'the')])

print("word_counts", word_counts['in'])

prob 0.2592470277410832
bigram_counts 785


In [86]:
transitions = {}

for (w1, w2), p in prob.items():
    if w1 not in transitions:
        transitions[w1] = []
    transitions[w1].append((w2, p))

transitions['in'][:10]


[('a', 0.10997357992073976),
 ('that', 0.023117569352708058),
 ('their', 0.005614266842800529),
 ('this', 0.01915455746367239),
 ('coarse', 0.0006605019815059445),
 ('water', 0.00033025099075297226),
 ('mud', 0.00033025099075297226),
 ('his', 0.062087186261558784),
 ('terror', 0.00033025099075297226),
 ('shore', 0.0006605019815059445)]

Ora posse generare il testo:

In [94]:
def generate_text(transitions, length, start=None):

    if start is None:
        start = rn.choice(list(transitions.keys()))

    new_text = [start]
    current_word = start

    for _ in range(length - 1):

        if current_word not in transitions:
            break
      
        ww = []
        pp = []
        for w, p in transitions[current_word]:
            ww.append(w)
            pp.append(p)
            
        nw = rn.choices(ww, weights = pp, k=1)[0]

        new_text.append(nw)
        current_word = nw

    return ' '.join(new_text)

new_text = generate_text(transitions, 50, 'in')

print(new_text)

in his gold plate and there no more dreadful casts of himself with some high wind and packages and our town and that you may be partners with the world more so but when it i call me afore the ditch tearing down dale in a small in herbert returned


Creiamo degli n-grammi:

In [ ]:
def create_ngrams(words, n):
    ngrams = []
    for i in range(len(words) - n + 1):
        ngram = tuple(words[i:i+n])
        ngrams.append(ngram)
    return ngrams

def calculate_transions(words, ngrams):
    
    ngrams_counts = Counter(ngrams)

    ngrams_minus1 = create_ngrams(words, n-1)
    ngrams_minus1_counts = Counter(ngrams_minus1)
    
    prob_cond = {}
    for ng in ngrams_counts:
        context = ng[:-1]  
        prob_cond[ng] = ngrams_counts[ng] / ngrams_minus1_counts[context]
        
    transitions = {}

    for ng, p in prob_cond.items():
        context = ng[:-1]
        if context not in transitions:
            transitions[context] = []
        transitions[context].append((ng[-1], p))
        
    return transitions

def generate_text_ngrams(transitions, length, start=None):

    if start is None:
        start = rn.choice(list(transitions.keys()))
    
    new_text = list(start)
    context = start

    for _ in range(length - len(start)):
        
        if context not in transitions:
            break

        ww = []
        pp = []
        for w, p in transitions[context]:
            ww.append(w)
            pp.append(p)

        nw = rn.choices(ww, weights=pp, k=1)[0]
        new_text.append(nw)
        context = tuple(new_text[-len(start):])  

    return ' '.join(new_text)

def generate_text(words,
                  n,
                  length):
    ngrams = create_ngrams(words, n)
    transitions = calculate_transions(words, ngrams)
    new_text = generate_text_ngrams(transitions, length, ('the',))

    print(f"testo generato con {n}-grammi")
    print(new_text)
    print("")



# bigrammi
n = 2
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)
new_text = generate_text_ngrams(transitions, 50, ('the',))

print("testo generato con bigrammi")
print(new_text)
print("")

# trigrammi
n = 3
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)
new_text = generate_text_ngrams(transitions, 50, ('the','morning'))

print("testo generato con trigrammi")
print(new_text)
print("")

# 4-grammi
n = 4
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)
new_text = generate_text_ngrams(transitions, 50, ('the','morning', 'i'))

print("testo generato con 4-grammi")
print(new_text)
print("")

testo generato con bigrammi
the house i have any pigeons and i had kept a tenderer bit of course of it come and think much apart so much changed now don t a foremost among the purpose whether to deal burned to claim me anywhere according to change at the last very dark complexioned

testo generato con trigrammi
the morning while my mind was too happy they were and how of living dear boy returned herbert and i might not prove unacceptabobble and biddy on this sunday when the bellows but by and with my whole entertainment nor was it not obvious that with the good sense of

testo generato con 4-grammi
the morning i brought it out and after slowly blowing all his smoke away and looking hard at me all the time upon him himself who would not be helped nor i extenuated a new fear had been engendered in my mind and heart of that reserved secondly which had



## Conditional entropy

$
    H(w_n | w_{n-1}) = - \sum_{w_{n-1}, w_n} P(w_{n-1}, w_n) \log_2 P(w_n | w_{n-1})
$

È un modo per misurare la predittibilità di un testo.

Conditional entropy gives you a map of language predictability.

Ad esempio dopo *I eat* posso avere:

* cake, 0.9
* bread, 0.05
* banana, 0.05

Questo testo ha un'entropia minore, è un testo molto predicibile.

Invece, se dopo *I eat* posso avere:

* cake, 0.4
* bread, 0.3
* banana, 0.3

Questo testo è meno predicibile: ha entropia maggiore.

Voglio riscalare le probabilità:

In [106]:
import math

def entropy(probs):
    return -sum(p * math.log2(p) for p in probs)

with open(file, 'r', encoding='utf-8') as f:
    text = f.read().lower()
    words = re.findall(r'\b[a-zàèéìòù]+\b', text)

n = 2
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)

transitions[('the',)][:10]

[('authority', 0.00024560972614515533),
 ('blacksmith', 0.0009824389045806213),
 ('days', 0.0014736583568709322),
 ('shape', 0.00024560972614515533),
 ('letters', 0.00012280486307257767),
 ('character', 0.00036841458921773305),
 ('inscription', 0.00012280486307257767),
 ('i', 0.00024560972614515533),
 ('memory', 0.00024560972614515533),
 ('marsh', 0.0008596340415080437)]

Sono i bigrammi per la parola `the`. Voglio introdurre una componente random, in modo da rendere il testo più "creativo".

In [114]:
entropy([p for w, p in transitions[('the',)]])

def apply_temperature(probs, T):
    
    adjusted_prob0 = []
    for w, p in probs:
        adjusted_prob0.append(math.pow(p, 1.0 / T))

    total = sum(adjusted_prob0)
    adjusted_prob = [(probs[i][0], adjusted_prob0[i]/total) for i in range(len(adjusted_prob0))]

    return adjusted_prob

The $p^{1/T}$ transformation comes from the maximum entropy principle, where temperature emerges as the Lagrange multiplier controlling the tradeoff between fitting constraints (log-probabilities) and maximizing uncertainty (entropy).

In [110]:
adjusted_prob = apply_temperature(transitions[('the',)], T=0.1)

print("Temperaure T=0.1")
print(adjusted_prob[:10])
print(entropy([p for w, p in adjusted_prob]))
print("")

adjusted_prob = apply_temperature(transitions[('the',)], T=10)

print("Temperaure T=10")
print(adjusted_prob[:10])
print(entropy([p for w, p in adjusted_prob]))

Temperaure T=0.1
[('authority', 3.3165737744114835e-18), ('blacksmith', 3.4776796620772958e-12), ('days', 2.0054053356054936e-10), ('shape', 3.3165737744114835e-18), ('letters', 3.2388415765737144e-21), ('character', 1.9125035625510155e-16), ('inscription', 3.2388415765737144e-21), ('i', 3.3165737744114835e-18), ('memory', 3.3165737744114835e-18), ('marsh', 9.148925808142132e-13)]
2.104781682563312

Temperaure T=10
[('authority', 0.00043038313185841677), ('blacksmith', 0.0004943803955842353), ('days', 0.0005148377297883776), ('shape', 0.00043038313185841677), ('letters', 0.00040156166102483886), ('character', 0.0004481922756733699), ('inscription', 0.00040156166102483886), ('i', 0.00043038313185841677), ('memory', 0.00043038313185841677), ('marsh', 0.00048782274536062945)]
11.181022827315111


Nel secondo caso l'entropia è minore.

Posso riscrivere la funzione per creare il testo introducendo la temperatura come parametro.

In [ ]:
def gen_text_ngrams_entropy(transitions, length, start=None, T=1):

    if start is None:
        start = rn.choice(list(transitions.keys()))
    
    new_text = list(start)
    context = start

    for _ in range(length - len(start)):
        
        if context not in transitions:
            break
        
        probs = transitions[context]
        probs_temp = apply_temperature(probs, T)
        
        ww = []
        pp = []
        for w, p in probs_temp:
            ww.append(w)
            pp.append(p)

        nw = rn.choices(ww, weights=pp, k=1)[0]
        new_text.append(nw)
        context = tuple(new_text[-len(start):])  
    
    return ' '.join(new_text)

n = 2
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)

T = 0.1
new_text = gen_text_ngrams_entropy(transitions, 50, ('the',), T=T)
print(f"testo generato con {n}-grammi e temperatura T={T}")
print(new_text)
print("")

testo generato con 2-grammi e temperatura T=0.1
the same time and i had been in the fire and i had been a little britain and i had been a little while i had been in the fire and i had been a little britain and i had been a little britain and i had been a little


Questo tipo di testo (ordine 0) può essere usato per generare testi tecnici.

Aumento la "creativitià" del modello:

In [113]:
n = 2
ngrams = create_ngrams(words, n)
transitions = calculate_transions(words, ngrams)

T = 10
new_text = gen_text_ngrams_entropy(transitions, 50, ('the',), T=T)
print(f"testo generato con {n}-grammi e temperatura T={T}")
print(new_text)
print("")

testo generato con 2-grammi e temperatura T=10
the normal perplexity of erudition i felt mortified to whom startop who always represented on a rubicund and cumbered with comfort to callings and drink including breakfast i turning delirious they measured and save my terrors of deadly cold hearths a company to its stabs and uncomfortable until we entered



Quando aumento la temperatura sto perdendo il significato perché sto riducendo la ridondanza di parole.

# Exercise
Build a Markov-chain-based generator trained on the scientific paper `paper.txt`, and analyze how temperature affects text quality.

When does the text look more “scientific”?

How does the output change if you add <START> and <END> tokens at the beginning and at the end of each sentence?